In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from bs4 import BeautifulSoup
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv('/kaggle/input/imdb-movie-reviews/IMDB Dataset.csv')
df

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
positive_reviews = df[df['sentiment'] == 'positive'].sample(n=5000, random_state=42)
negative_reviews = df[df['sentiment'] == 'negative'].sample(n=5000, random_state=42)

df = pd.concat([positive_reviews, negative_reviews]).sample(frac=1.0, random_state=42).reset_index(drop=True)


In [ ]:
plt.figure(figsize=(4,6))
sns.countplot(data=df,x='sentiment')
plt.show()

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.duplicated().sum()

In [ ]:
df['review'].head()

In [ ]:
def clean_text(text):
    
    text=BeautifulSoup(text,'html.parser').get_text()
    text=re.sub(r"[^A-Za-z0-9\s']","",text)
    text=text.lower()
    return text

cleaned_reviews=df['review'].apply(clean_text)

print(cleaned_reviews.head())

In [ ]:
import tensorflow as tf
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [ ]:
#i figured this out:
 
"""Stopwords → Transformer models like DistilBERT don’t require stopword removal. 
Stopwords can carry context, so removing them can actually hurt performance.""" 

# and also this:
#DistilBERT already uses WordPiece tokenization — words are split into subword units, so related forms share pieces.

In [ ]:

def prepare_bert_data(reviews,sentiments):
    
    labels=LabelEncoder().fit_transform(sentiments)
    
    x_train,x_test,y_train,y_test=train_test_split(reviews,labels, test_size=0.2, stratify=labels,random_state=42)
    
    tokenizer=AutoTokenizer.from_pretrained('distilbert-base-uncased')
    
    train_encodings = tokenizer(x_train, truncation=True, padding='max_length', max_length=250)
    test_encodings = tokenizer(x_test, truncation=True, padding='max_length', max_length=250)
    
    train_dataset=tf.data.Dataset.from_tensor_slices((dict(train_encodings), y_train)).shuffle(len(y_train)).batch(32)
    test_dataset=tf.data.Dataset.from_tensor_slices((dict(test_encodings), y_test)).batch(32)

    return train_dataset, test_dataset, tokenizer, y_test






In [ ]:
from transformers import TFDistilBertForSequenceClassification, create_optimizer
from tensorflow.keras.losses import SparseCategoricalCrossentropy

In [ ]:
def bert_model(train_dataset):

    steps_per_epoch = tf.data.experimental.cardinality(train_dataset).numpy()
    epochs=2
    num_train_steps=steps_per_epoch*epochs
    
    
    optimizer,_=create_optimizer(
        init_lr=2e-5,
        num_train_steps=num_train_steps,
        num_warmup_steps=int(0.1*num_train_steps)
    )
    
    model=TFDistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

    
    model.compile(
        optimizer=optimizer,
        loss=SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    
    model.summary()

    return model
    

In [ ]:
def train_history_for_bert(bert_model,train_dataset,test_dataset):
    history=bert_model.fit(
        train_dataset,
        validation_data=test_dataset,
        epochs=2
    )
    return history

In [ ]:
reviews=df['review'].tolist()
sentiments=df['sentiment']



In [ ]:
train_dataset,test_dataset,tokenizer,y_test=prepare_bert_data(reviews,sentiments)
model=bert_model(train_dataset)
model_train_history=train_history_for_bert(model,train_dataset,test_dataset)


In [ ]:
save_path = "/kaggle/working/bert_sentiment_nada"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)